# Rolling windows vs exponential weighting

screamer computes the same statistics two ways. A **rolling window** averages the
last *n* samples with equal weight and a fixed lookback. **Exponential weighting**
keeps all history but lets it decay geometrically, so recent samples count more.

The trade-off is responsiveness against stability: exponential weighting turns
with the data sooner, while a rolling window is steadier and forgets on a fixed
schedule. Reach for a rolling window when you want a hard, interpretable lookback;
reach for exponential weighting when you want faster response without choosing a
cutoff. The same choice applies across the family: mean, std, z-score, and the
rest.

Both share the one calling convention from
[notebook 1](01-quickstart-polymorphic-api.ipynb) and run on live streams; the
rolling family also matches the pandas numbers you already know.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import RollingMean, RollingStd, EwMean, EwStd

rng = np.random.default_rng(0)
price = 100 + np.cumsum(rng.standard_normal(300))

## The same trade-off, for level and for volatility

With a matched window and span of 20, the rolling and exponential versions track
the series differently. In the level (top), the exponential mean hugs recent
moves while the rolling mean lags. In the volatility (bottom), the exponential
std reacts to a burst of movement sooner and relaxes more smoothly.

In [ ]:
roll_mean = RollingMean(20)(price)
ew_mean   = EwMean(span=20)(price)
roll_std  = RollingStd(20)(price)
ew_std    = EwStd(span=20)(price)

fig, (ax_level, ax_vol) = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

ax_level.plot(price, lw=0.6, color="0.6", label="price")
ax_level.plot(roll_mean, label="RollingMean(20)")
ax_level.plot(ew_mean, label="EwMean(span=20)")
ax_level.set_ylabel("level")
ax_level.legend(loc="upper left")

ax_vol.plot(roll_std, label="RollingStd(20)")
ax_vol.plot(ew_std, label="EwStd(span=20)")
ax_vol.set_ylabel("volatility")
ax_vol.set_xlabel("sample")
ax_vol.legend(loc="upper left")

plt.tight_layout()

## Warm-up differs

A rolling window has nothing to report until it is full, so `RollingMean(20)`
emits `NaN` for the first 19 samples. Exponential weighting has a value from the
very first sample, though it takes roughly a span to settle. You can change the
rolling window's warm-up with the `start_policy` argument; see
[NaN and warmup](../nan_and_warmup.md).

In [ ]:
print("rolling first 3:", roll_mean[:3].round(3))   # NaN until the window fills
print("ew first 3     :", ew_mean[:3].round(3))     # a value from sample 0

## Same numbers as pandas

The rolling family is numerically identical to pandas, so you keep the results
you know and gain streaming and speed. Exponential-weighting conventions vary
between libraries, so the EW family is not a drop-in for pandas `ewm`; see
[conventions](../conventions.md) for the details.

In [ ]:
import pandas as pd   # only needed to check equivalence, not to use screamer

pd_mean = pd.Series(price).rolling(20).mean().to_numpy()
np.testing.assert_allclose(roll_mean, pd_mean, equal_nan=True)
print("RollingMean matches pandas rolling(20).mean()")

## The rolling and EW families

The two families mirror each other for the moment-based statistics. Where a cell
is empty, that statistic has no exponentially-weighted form (a decayed median or
maximum has no clean meaning).

| statistic | Rolling | Exponentially-weighted |
|---|---|---|
| mean | [`RollingMean`](../functions_rolling/RollingMean.md) | [`EwMean`](../functions_ew/EwMean.md) |
| std / var | [`RollingStd`](../functions_rolling/RollingStd.md), [`RollingVar`](../functions_rolling/RollingVar.md) | [`EwStd`](../functions_ew/EwStd.md), [`EwVar`](../functions_ew/EwVar.md) |
| z-score | [`RollingZscore`](../functions_rolling/RollingZscore.md) | [`EwZscore`](../functions_ew/EwZscore.md) |
| skew / kurtosis | [`RollingSkew`](../functions_rolling/RollingSkew.md), [`RollingKurt`](../functions_rolling/RollingKurt.md) | [`EwSkew`](../functions_ew/EwSkew.md), [`EwKurt`](../functions_ew/EwKurt.md) |
| rms | [`RollingRms`](../functions_rolling/RollingRms.md) | [`EwRms`](../functions_ew/EwRms.md) |
| correlation / covariance | [`RollingCorr`](../functions_fin/RollingCorr.md) | [`EwCorr`](../functions_ew/EwCorr.md), [`EwCov`](../functions_ew/EwCov.md), [`EwBeta`](../functions_ew/EwBeta.md) |
| median / quantile | [`RollingMedian`](../functions_rolling/RollingMedian.md), [`RollingQuantile`](../functions_rolling/RollingQuantile.md) | |
| min / max | [`RollingMin`](../functions_rolling/RollingMin.md), [`RollingMax`](../functions_rolling/RollingMax.md) | |
| sum | [`RollingSum`](../functions_rolling/RollingSum.md) | |

For the complete list, including the financial indicators and OHLC volatility
estimators built on rolling windows, see the
[function reference](../by_topic_index.rst).